# Congested Potential-Field Reward Test

This notebook tests one fixed setup:

**Attention DQN + base highway reward + potential-field proximity penalty**

It replaces the old reward-safety-factor sweep. The potential-field idea is adapted from the laneless Karalakou notebook, but only the proximity field is used here. The full Karalakou reward, lateral zone reward, overtake bonus, adaptive TTC controller, TTC observation, and lane-change safety penalty are intentionally left out so this first test is clean.

## Imports And Paths

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
RESULTS_SUBDIR = "ct_pf"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "dqn" / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT / "src" / "deep_learning" / "DQN", PROJECT_ROOT / "notebooks" / "_shared"]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import dqn_notebook_utils

dqn_notebook_utils = importlib.reload(dqn_notebook_utils)
from dqn_notebook_utils import (
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    train_and_display,
)

print("Project root:", PROJECT_ROOT)
print("Results dir:", RESULTS_DIR)

## Experiment Setup

In [ ]:
environment_profile = "structured_baseline"

congested_environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 30,
    "duration": 40,
    "ego_spacing": 1.8,
    "vehicles_density": 1.2,
    "simulation_frequency": 15,
    "policy_frequency": 3,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}

congested_observation_config = {
    "vehicles_count": 12,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}

action_config = {
    "type": "DiscreteMetaAction",
}

base_reward_config = {
    "collision_reward": -5.0,
    "right_lane_reward": 0.03,
    "high_speed_reward": 0.25,
    "lane_change_reward": -0.02,
    "normalize_reward": False,
}

speed_config = {
    "reward_speed_range": [15.0, 28.0],
}

congested_driver_aggressiveness_config = {
    "enabled": True,
    "distribution": "uniform",
    "min_score": 0.0,
    "max_score": 100.0,
    "fixed_score": None,
    "normal_mean": 50.0,
    "normal_std": 20.0,
    "conservative": {
        "target_speed": 18.0,
        "acc_max": 4.0,
        "comfort_acc_max": 2.0,
        "comfort_acc_min": -4.0,
        "delta": 4.5,
        "time_wanted": 2.4,
        "distance_wanted": 14.0,
        "politeness": 0.8,
        "lane_change_min_acc_gain": 0.8,
        "lane_change_max_braking_imposed": 1.0,
        "lane_change_delay": 1.5,
    },
    "aggressive": {
        "target_speed": 30.0,
        "acc_max": 7.0,
        "comfort_acc_max": 5.5,
        "comfort_acc_min": -6.5,
        "delta": 3.5,
        "time_wanted": 0.6,
        "distance_wanted": 6.0,
        "politeness": 0.0,
        "lane_change_min_acc_gain": 0.05,
        "lane_change_max_braking_imposed": 3.5,
        "lane_change_delay": 0.5,
    },
}

potential_field_reward_config = {
    "enabled": True,
    "weight": 0.25,
    "sensing_range": 120.0,
    "field_magnitude": 0.5,
    "field_px": 2.0,
    "field_py": 2.0,
    "field_pt": 1.0,
    "timegap": 0.7,
    "lateral_timegap": 0.7,
    "max_cost": 1.0,
    "lanes": "ego_and_adjacent",
}

attention_config = {
    "features_dim": 64,
    "attention_heads": 2,
    "attention_dropout": 0.0,
    "presence_feature_idx": 0,
    "embedding_arch": "64,64",
    "net_arch": "64,64",
}

timesteps = 20_000
eval_episodes_during_training = 10
saved_model_eval_episodes = 1000
n_envs = min(4, os.cpu_count() or 1)
seed = 42
train_freq = 4
gradient_steps = 4
saved_model_eval_seed = seed + 10_000
saved_model_eval_name = f"eval{saved_model_eval_episodes}"

hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.40,
    "exploration_initial_eps": 1.0,
    "exploration_final_eps": 0.05,
    "progress_every": 1_000,
    "verbose": 0,
}

display(pd.DataFrame({"potential_field_reward_config": pd.Series(potential_field_reward_config)}))

## Build Env Config

In [ ]:
def make_potential_field_env_config() -> dict:
    return build_env_config(
        profile_name=environment_profile,
        profile_overrides=congested_environment_overrides,
        observation=congested_observation_config,
        action=action_config,
        reward=base_reward_config,
        speed=speed_config,
        adaptive_longitudinal={"enabled": False},
        rear_flow={"enabled": False},
        traffic_flow_reward={"enabled": False},
        safety_ttc_flow_reward={"enabled": False},
        potential_field_reward=potential_field_reward_config,
        driver_aggressiveness=congested_driver_aggressiveness_config,
        driver_aggressiveness_observation={"enabled": False},
        ttc_observation={"enabled": False},
        lane_change_safety={"enabled": False},
    )


env_config = make_potential_field_env_config()
display(pd.DataFrame({"env_config": pd.Series(env_config)}))

## Smoke Test The Reward Wrapper

In [ ]:
trainer, _, _, _, default_device = load_dqn_backend(
    backend_module="attention_dqn",
    notebook_subdir="congested_traffic_policy",
    results_subdir=RESULTS_SUBDIR,
)

env = trainer.make_env(render_mode=None, config=env_config)
obs, info = env.reset(seed=seed)
rows = []
for step in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    rows.append({
        "step": step,
        "reward": float(reward),
        "potential_field_cost": float(info.get("potential_field_cost", 0.0)),
        "potential_field_penalty": float(info.get("potential_field_penalty", 0.0)),
        "vehicle_count": float(info.get("potential_field_vehicle_count", 0.0)),
        "closest_longitudinal_gap": float(info.get("potential_field_closest_longitudinal_gap", float("nan"))),
        "closest_lateral_gap": float(info.get("potential_field_closest_lateral_gap", float("nan"))),
    })
    if terminated or truncated:
        break
env.close()

display(pd.DataFrame(rows).round(3))

## Train And Evaluate

In [ ]:
run_name = "attn_pf20k"
label = "Attention DQN + Base Reward + Potential Field"

trainer, _, _, results_dir, default_device = load_dqn_backend(
    backend_module="attention_dqn",
    notebook_subdir="congested_traffic_policy",
    results_subdir=RESULTS_SUBDIR,
)

args = build_dqn_args(
    results_dir=results_dir,
    run_name=run_name,
    timesteps=timesteps,
    eval_episodes=eval_episodes_during_training,
    seed=seed,
    num_envs=n_envs,
    device=default_device,
    hyperparameters=hyperparameters,
    extra=attention_config,
)

summary = train_and_display(trainer, args, env_config, label)
saved_eval_summary = evaluate_saved_model(
    trainer,
    summary_path=results_dir / run_name / "summary.json",
    env_config=env_config,
    episodes=saved_model_eval_episodes,
    seed=saved_model_eval_seed,
    name=saved_model_eval_name,
    label=label,
)

output = {
    "run_name": run_name,
    "label": label,
    "backend_module": "attention_dqn",
    "potential_field_weight": potential_field_reward_config["weight"],
    "summary": summary,
    "saved_eval_summary": saved_eval_summary,
}

comparison_rows = []
row = {
    "run_name": run_name,
    "label": label,
    "potential_field_weight": potential_field_reward_config["weight"],
}
for record in saved_eval_summary.get("metric_summary", []):
    metric = (
        record.get("metric", "")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_per_")
        .lower()
    )
    row[f"{metric}_mean"] = record.get("mean")
    row[f"{metric}_std"] = record.get("std")
comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = RESULTS_DIR / "summary.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Comparison saved to:", comparison_path)
display(comparison_df.round(3))

## Load Existing Result

In [ ]:
comparison_path = RESULTS_DIR / "summary.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path).round(3))
else:
    print("No saved result yet:", comparison_path)